### STEP 1 — Create clean working dataframe

In [86]:
import pandas as pd 
import numpy as np 
import warnings
warnings.filterwarnings('ignore')

df=pd.read_excel(r'data\visits for an-april-2026.xlsx')
df.head()


,CLAIM ID,CENTRAL ID,CLAIM TYPE,SCHEME,MEMBER NUMBER,INTEG MEMBER NUMBER,OTHER NUMBER,OFFICE BRANCH,CARD SERIAL,PATIENT NAME,...,MEMBER SPLIT AMOUNT,GLOBAL INVOICE NUMBER,PRINCIPAL NAMES,PRINCIPAL MEMBER NUMBER,PRINCIPAL OTHER NUMBER,RELATIONSHIP TO PRINCIPAL,PHONE NUMBER,PHONE NUMBER2,PARENT POOL,PARENT POOL DESCRIPTION
0,469509392,387567048,Normal claim,DEFENCE FORCES MEDICAL INSURANCE SCHEME,DEFMIS-88090-02,2511314.0,88090,NaN,VCKE0001799008,NIA KEMI ABBEY,...,0.0,AAR261900945,NaN,NaN,NaN,NaN,254759622720,NaN,NaN,NaN
1,469413003,387438120,Normal claim,DEFENCE FORCES MEDICAL INSURANCE SCHEME,DEFMIS-20998-02,2023278.0,20998,NaN,VCKE0001607696,RYAN GATHUO NJOROGE,...,0.0,AAR261900755,NaN,NaN,NaN,NaN,254722496558,NaN,NaN,NaN
2,470786070,403862518,Normal claim,DEFENCE FORCES MEDICAL INSURANCE SCHEME,DEFMIS-20998-02,2023278.0,20998,NaN,VCKE0001607696,RYAN GATHUO NJOROGE,...,0.0,AAR261903053,NaN,NaN,NaN,NaN,254722496558,NaN,NaN,NaN
3,469582181,387659720,Normal claim,DEFENCE FORCES MEDICAL INSURANCE SCHEME,DEFMIS-18898-03,2016300.0,18898,NaN,VCKE0001606846,ABDULMUQQIT ADOW OSMAN,...,0.0,AAR261901080,NaN,NaN,NaN,NaN,254729444900,NaN,NaN,NaN
4,469433479,387464741,Normal claim,DEFENCE FORCES MEDICAL INSURANCE SCHEME,DEFMIS-18898-03,2016300.0,18898,NaN,VCKE0001606846,ABDULMUQQIT ADOW OSMAN,...,0.0,AAR261900799,NaN,NaN,NaN,NaN,254729444900,NaN,NaN,NaN


### STEP 2 — Standardize columns

In [87]:
df.columns = df.columns.str.strip()

In [88]:
X = df.shape
print(f"The dataset has {X[0]} rows and {X[1]} columns.")

The dataset has 22944 rows and 43 columns.


### STEP 3 — Ensure correct data types

In [89]:
df['AMOUNT'] = pd.to_numeric(
    df['AMOUNT'],
    errors='coerce'
)

df['ARRIVAL DATE'] = pd.to_datetime(
    df['ARRIVAL DATE']
)

In [90]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22944 entries, 0 to 22943
Data columns (total 43 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   CLAIM ID                   22944 non-null  int64         
 1   CENTRAL ID                 22944 non-null  int64         
 2   CLAIM TYPE                 22944 non-null  object        
 3   SCHEME                     22944 non-null  object        
 4   MEMBER NUMBER              22944 non-null  object        
 5   INTEG MEMBER NUMBER        22943 non-null  float64       
 6   OTHER NUMBER               22944 non-null  object        
 7   OFFICE BRANCH              39 non-null     object        
 8   CARD SERIAL                22754 non-null  object        
 9   PATIENT NAME               22944 non-null  object        
 10  DOB                        22944 non-null  datetime64[ns]
 11  CAT CODE                   22944 non-null  object        
 12  CAT 

In [91]:
service_cost = (
    df.groupby('SERVICE TYPE')['AMOUNT']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

service_cost.columns = ['SERVICE TYPE', 'TOTAL COST']
service_cost['TOTAL COST'] = service_cost['TOTAL COST'].apply(lambda x: f"{x:,.2f}")
service_cost.head()

,SERVICE TYPE,TOTAL COST
0,IP,"184,946,092.08"
1,OP,"140,596,161.32"


### STEP 4 — Create TRUE VISIT LOGIC

`This is important for utilization analysis.`

`One member + one arrival date = one visit.`

In [92]:
df['VISIT_KEY'] = (
    df['MEMBER NUMBER'].astype(str)
    + '_'
    + df['ARRIVAL DATE'].dt.strftime('%Y-%m-%d')
    + '_'
    + df['SERVICE TYPE'].astype(str)
)

### STEP 5 — Create your FIRST clean summary table


`We start simple:`

- SERVICE TYPE
- UNIQUE VISITS
- TOTAL COST

In [93]:
service_summary = df.groupby('SERVICE TYPE').agg(
    UNIQUE_VISITS=('VISIT_KEY', 'nunique'),
    TOTAL_COST=('AMOUNT', 'sum')
).reset_index()
service_summary

,SERVICE TYPE,UNIQUE_VISITS,TOTAL_COST
0,IP,1483,1.849461e+08
1,OP,18331,1.405962e+08


In [94]:
benefit_summary = df.groupby('BENEFIT DESC').agg(
    UNIQUE_VISITS=('VISIT_KEY', 'nunique'),
    TOTAL_COST=('AMOUNT', 'sum')
).reset_index()

In [95]:
benefit_summary['AVG_COST_PER_VISIT'] = (
    benefit_summary['TOTAL_COST']
    / benefit_summary['UNIQUE_VISITS']
).round(2)
benefit_summary['TOTAL_COST'] = benefit_summary['TOTAL_COST'].apply(lambda x: f"{x:,.2f}")
benefit_summary['AVG_COST_PER_VISIT'] = benefit_summary['AVG_COST_PER_VISIT'].apply(lambda x: f"{x:,.2f}")
benefit_summary

,BENEFIT DESC,UNIQUE_VISITS,TOTAL_COST,AVG_COST_PER_VISIT
0,IN PATIENT OVERALL / HOSPITALIZATION/ACCOMODATION,1483,"184,946,092.08","124,710.78"
1,OUT PATIENT DENTAL,425,"3,194,792.58","7,517.16"
2,OUT PATIENT OPTICAL,629,"6,474,724.72","10,293.68"
3,OUT PATIENT OVERALL,17502,"130,926,644.02","7,480.67"


### TEP 6 — Add average cost per visit

In [96]:
service_summary['AVG_COST_PER_VISIT'] = (
    service_summary['TOTAL_COST']
    / service_summary['UNIQUE_VISITS']
).round(2)
service_summary


,SERVICE TYPE,UNIQUE_VISITS,TOTAL_COST,AVG_COST_PER_VISIT
0,IP,1483,1.849461e+08,124710.78
1,OP,18331,1.405962e+08,7669.86


### STEP 7 — Add percentage contribution

In [97]:
grand_total = service_summary['TOTAL_COST'].sum()

service_summary['PERCENT_OF_TOTAL'] = (
    service_summary['TOTAL_COST']
    / grand_total * 100
).round(2)
grand_total

325542253.40040004

In [98]:
service_summary_display = service_summary.copy()

service_summary_display['TOTAL_COST'] = (
    service_summary_display['TOTAL_COST']
    .apply(lambda x: f"{x:,.2f}")
)

service_summary_display['AVG_COST_PER_VISIT'] = (
    service_summary_display['AVG_COST_PER_VISIT']
    .apply(lambda x: f"{x:,.2f}")
)

service_summary_display['UNIQUE_VISITS'] = (
    service_summary_display['UNIQUE_VISITS']
    .apply(lambda x: f"{x:,}")
)

service_summary_display['PERCENT_OF_TOTAL'] = (
    service_summary_display['PERCENT_OF_TOTAL']
    .astype(str) + '%'
)
service_summary_display

,SERVICE TYPE,UNIQUE_VISITS,TOTAL_COST,AVG_COST_PER_VISIT,PERCENT_OF_TOTAL
0,IP,"1,483","184,946,092.08","124,710.78",56.81%
1,OP,"18,331","140,596,161.32","7,669.86",43.19%


In [99]:
pivot_table = pd.pivot_table(
    df,
    index='BENEFIT DESC',
    columns='SERVICE TYPE',
    values='AMOUNT',
    aggfunc='sum',
    fill_value=0
)


In [100]:
pivot_table['TOTAL'] = pivot_table.sum(axis=1)
pivot_table.loc['TOTAL'] = pivot_table.sum()
pivot_table = pivot_table.sort_values(
    by='TOTAL',
    ascending=True
)
pivot_table['TOTAL']= pivot_table['TOTAL'].apply(lambda x: f"{x:,.2f}")
pivot_table

SERVICE TYPE,IP,OP,TOTAL
BENEFIT DESC,,,
OUT PATIENT DENTAL,0.000000e+00,3.194793e+06,"3,194,792.58"
OUT PATIENT OPTICAL,0.000000e+00,6.474725e+06,"6,474,724.72"
OUT PATIENT OVERALL,0.000000e+00,1.309266e+08,"130,926,644.02"
IN PATIENT OVERALL / HOSPITALIZATION/ACCOMODATION,1.849461e+08,0.000000e+00,"184,946,092.08"
TOTAL,1.849461e+08,1.405962e+08,"325,542,253.40"


###      Provider Intelligence Layer (Core of Fraud System)

This is where the real value is.

In [101]:
provider_benefit_pivot = pd.pivot_table(
    df,
    index='MAIN HOSPITAL',
    columns='BENEFIT DESC',
    values='AMOUNT',
    aggfunc='sum',
    fill_value=0
)


In [102]:
provider_benefit_pivot['TOTAL_COST'] = provider_benefit_pivot.sum(axis=1)

In [103]:
provider_benefit_pivot = provider_benefit_pivot.sort_values(
    by='TOTAL_COST',
    ascending=False
)

In [104]:
provider_benefit_pivot.loc['TOTAL'] = provider_benefit_pivot.sum().round(2)
provider_benefit_pivot['TOTAL_COST'] = provider_benefit_pivot['TOTAL_COST'].apply(lambda x: f"{x:,.2f}")
provider_benefit_pivot

BENEFIT DESC,IN PATIENT OVERALL / HOSPITALIZATION/ACCOMODATION,OUT PATIENT DENTAL,OUT PATIENT OPTICAL,OUT PATIENT OVERALL,TOTAL_COST
MAIN HOSPITAL,,,,,
ULINZI PRIME HEALTH SERVICES FUND (UPHSF),2.342927e+07,801304.53,34494.81,3.369147e+07,"57,956,543.64"
NAIROBI HOSP REFERRAL,1.392582e+07,62489.00,0.00,6.485377e+06,"20,473,685.33"
NAIROBI WEST HOSP,1.516553e+07,75789.00,0.00,3.660974e+06,"18,902,288.96"
THE KAREN HOSP REFERRAL,1.089267e+07,42932.00,21000.00,3.934186e+06,"14,890,788.60"
ST LUKE ORTHOPEADIC ELD,1.177323e+07,38841.00,32566.00,2.719003e+06,"14,563,644.36"
...,...,...,...,...,...
MEDINA DIAGONOSTICS HOLA,0.000000e+00,0.00,0.00,6.350000e+03,"6,350.00"
MATATA NURSING HOME,0.000000e+00,0.00,0.00,3.765000e+03,"3,765.00"
TEXAS CANCER CENTRE LTD,0.000000e+00,0.00,0.00,2.625000e+03,"2,625.00"


In [105]:
# Assuming df is already loaded, cleaned, and has feature columns (TRANSACTION DATE, VISIT_KEY)
# If you haven't created VISIT_KEY yet, run:
# df['VISIT_KEY'] = df['MEMBER NUMBER'].astype(str) + '_' + df['TRANSACTION DATE'].dt.strftime('%Y-%m-%d') + '_' + df['SERVICE TYPE'].astype(str)

# 1. Basic monthly aggregates
monthly_summary = df.groupby(df['TRANSACTION DATE'].dt.to_period('M')).agg(
    TOTAL_CLAIMS=('CLAIM ID', 'count'),
    TOTAL_AMOUNT=('AMOUNT', 'sum'),
    UNIQUE_VISITS=('VISIT_KEY', 'nunique')
).reset_index()

monthly_summary['YEAR_MONTH'] = monthly_summary['TRANSACTION DATE'].astype(str)
#print("=== Monthly Summary ===")
#print(monthly_summary.to_string(index=False))

# 2. Monthly breakdown by SERVICE TYPE and BENEFIT DESC
monthly_breakdown = df.groupby([
    df['TRANSACTION DATE'].dt.to_period('M'),
    'SERVICE TYPE',
    'BENEFIT DESC'
]).agg(
    CLAIMS=('CLAIM ID', 'count'),
    TOTAL_AMOUNT=('AMOUNT', 'sum'),
    UNIQUE_VISITS=('VISIT_KEY', 'nunique')
).reset_index()

monthly_breakdown['YEAR_MONTH'] = monthly_breakdown['TRANSACTION DATE'].astype(str)
monthly_breakdown = monthly_breakdown.drop('TRANSACTION DATE', axis=1)

#print("\n=== Monthly Breakdown by Service Type & Benefit Desc ===")
#print(monthly_breakdown.to_string(index=False))

# Optional: Pivot table for easier reading
pivot_amount = monthly_breakdown.pivot_table(
    index=['YEAR_MONTH', 'SERVICE TYPE'],
    columns='BENEFIT DESC',
    values='TOTAL_AMOUNT',
    aggfunc='sum',
    fill_value=0
)
monthly_summary



,TRANSACTION DATE,TOTAL_CLAIMS,TOTAL_AMOUNT,UNIQUE_VISITS,YEAR_MONTH
0,2026-01,5936,7.515678e+07,5083,2026-01
1,2026-02,5593,8.852881e+07,4835,2026-02
2,2026-03,5755,8.692050e+07,4981,2026-03
3,2026-04,5660,7.493616e+07,4936,2026-04


In [106]:
monthly_breakdown

,SERVICE TYPE,BENEFIT DESC,CLAIMS,TOTAL_AMOUNT,UNIQUE_VISITS,YEAR_MONTH
0,IP,IN PATIENT OVERALL / HOSPITALIZATION/ACCOMODATION,386,3.764328e+07,372,2026-01
1,OP,OUT PATIENT DENTAL,113,8.608979e+05,105,2026-01
2,OP,OUT PATIENT OPTICAL,176,1.787354e+06,173,2026-01
3,OP,OUT PATIENT OVERALL,5261,3.486525e+07,4498,2026-01
4,IP,IN PATIENT OVERALL / HOSPITALIZATION/ACCOMODATION,424,5.471028e+07,404,2026-02
5,OP,OUT PATIENT DENTAL,108,8.150049e+05,103,2026-02
6,OP,OUT PATIENT OPTICAL,182,1.712176e+06,169,2026-02
7,OP,OUT PATIENT OVERALL,4879,3.129135e+07,4203,2026-02
8,IP,IN PATIENT OVERALL / HOSPITALIZATION/ACCOMODATION,373,5.206192e+07,368,2026-03
9,OP,OUT PATIENT DENTAL,116,7.755148e+05,113,2026-03


In [107]:
pivot_amount

BENEFIT DESC             IN PATIENT OVERALL / HOSPITALIZATION/ACCOMODATION  \
YEAR_MONTH SERVICE TYPE                                                      
2026-01    IP                                                  37643281.40   
           OP                                                         0.00   
2026-02    IP                                                  54710280.70   
           OP                                                         0.00   
2026-03    IP                                                  52061915.26   
           OP                                                         0.00   
2026-04    IP                                                  40530614.72   
           OP                                                         0.00   

BENEFIT DESC             OUT PATIENT DENTAL  OUT PATIENT OPTICAL  \
YEAR_MONTH SERVICE TYPE                                            
2026-01    IP                          0.00                 0.00   
           OP                     860897.88           1787353.59   
2026-02    IP                          0.00                 0.00   
           OP                     815004.86           1712176.16   
2026-03    IP                          0.00                 0.00   
           OP                     775514.84           1587603.93   
2026-04    IP                          0.00                 0.00   
           OP                     743375.00           1387591.04   

BENEFIT DESC             OUT PATIENT OVERALL  
YEAR_MONTH SERVICE TYPE                       
2026-01    IP                   0.000000e+00  
           OP                   3.486525e+07  
2026-02    IP                   0.000000e+00  
           OP                   3.129135e+07  
2026-03    IP                   0.000000e+00  
           OP                   3.249547e+07  
2026-04    IP                   0.000000e+00  
           OP                   3.227458e+07

In [108]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Assuming you already have your DataFrame `df` loaded and cleaned

# ==========================================
# MONTHLY TREND VISUAL (Jan, Feb, Mar, ...)
# ==========================================

# 1. Create month-year and month name columns
df['MONTH_NUM'] = df['TRANSACTION DATE'].dt.month
df['MONTH_NAME'] = df['TRANSACTION DATE'].dt.strftime('%b')  # Jan, Feb, Mar, ...
df['YEAR_MONTH_SORT'] = df['TRANSACTION DATE'].dt.to_period('M')

# 2. Aggregate by month (chronological order)
monthly = df.groupby(['YEAR_MONTH_SORT', 'MONTH_NUM', 'MONTH_NAME']).agg(
    TOTAL_AMOUNT=('AMOUNT', 'sum'),
    TOTAL_CLAIMS=('CLAIM ID', 'count'),
    UNIQUE_VISITS=('VISIT_KEY', 'nunique')
).reset_index()

# Sort by actual date
monthly = monthly.sort_values('YEAR_MONTH_SORT')

# 3. Create the figure
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add traces
fig.add_trace(
    go.Scatter(x=monthly['MONTH_NAME'], y=monthly['TOTAL_AMOUNT'],
               name='💰 Total Amount (KSH)', mode='lines+markers',
               line=dict(color='blue', width=3), marker=dict(size=8)),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(x=monthly['MONTH_NAME'], y=monthly['TOTAL_CLAIMS'],
               name='📄 Total Claims', mode='lines+markers',
               line=dict(color='orange', width=3), marker=dict(size=8)),
    secondary_y=True
)

fig.add_trace(
    go.Scatter(x=monthly['MONTH_NAME'], y=monthly['UNIQUE_VISITS'],
               name='👥 Unique Visits', mode='lines+markers',
               line=dict(color='green', width=3, dash='dot'), marker=dict(size=8)),
    secondary_y=True
)

# Layout
fig.update_xaxes(title_text="Month", tickangle=0)
fig.update_yaxes(title_text="Amount (KSH)", secondary_y=False)
fig.update_yaxes(title_text="Count (Claims / Visits)", secondary_y=True)
fig.update_layout(
    title="📅 Monthly Trends: Amount, Claims & Unique Visits",
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=500
)

# Show the figure (works in Jupyter, VS Code, or any Python environment)
fig.show()

In [109]:
import plotly.express as px

# ==========================================
# MONTHLY BAR CHART - IMPROVED COLOR
# ==========================================

# 1. Prepare monthly aggregates
df['MONTH_NAME'] = df['TRANSACTION DATE'].dt.strftime('%b')
df['YEAR_MONTH_SORT'] = df['TRANSACTION DATE'].dt.to_period('M')

monthly_amount = df.groupby(['YEAR_MONTH_SORT', 'MONTH_NAME'])['AMOUNT'].sum().reset_index()
monthly_amount = monthly_amount.sort_values('YEAR_MONTH_SORT')

# 2. Create bar chart with better color
fig = px.bar(
    monthly_amount, 
    x='MONTH_NAME', 
    y='AMOUNT',
    title='💰 Monthly Total Amount',
    labels={'MONTH_NAME': 'Month', 'AMOUNT': 'Total Amount (Ksh)'},
    text_auto='.2s',
    color='AMOUNT',
    color_continuous_scale='Viridis',
)

# 3. Improve layout
fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Amount (Ksh)",
    height=450,
    showlegend=False,
    font=dict(size=12),
    plot_bgcolor='white',
    xaxis=dict(gridcolor='lightgray'),
    yaxis=dict(gridcolor='lightgray')
)

# ✅ CORRECT METHOD NAMES (plural)
fig.update_yaxes(tickformat=',.0f')
fig.update_traces(texttemplate='Ksh %{y:,.0f}', textposition='outside')

fig.show()

In [110]:
import pandas as pd
import plotly.express as px

# ==========================================
# SERVICE TYPE SUMMARY (NOTEBOOK VERSION)
# ==========================================

# 1. Aggregate by SERVICE TYPE
service_summary = df.groupby('SERVICE TYPE').agg(
    TOTAL_AMOUNT=('AMOUNT', 'sum'),
    UNIQUE_VISITS=('VISIT_KEY', 'nunique')     # unique visit = member + date + service type
).reset_index()

# Format amount in millions (e.g., 75.6M)
service_summary['AMOUNT_MILLIONS'] = service_summary['TOTAL_AMOUNT'] / 1_000_000
service_summary['AMOUNT_LABEL'] = service_summary['AMOUNT_MILLIONS'].apply(lambda x: f'{x:.1f}M')

# Create custom hover text that includes both amount and unique visits
service_summary['HOVER_TEXT'] = service_summary.apply(
    lambda row: f"Service: {row['SERVICE TYPE']}<br>Amount: {row['AMOUNT_LABEL']}<br>Unique Visits: {row['UNIQUE_VISITS']:,}",
    axis=1
)

# 2. Create bar chart
fig = px.bar(
    service_summary,
    x='SERVICE TYPE',
    y='TOTAL_AMOUNT',
    title='💰 Total Amount by Service Type',
    labels={'SERVICE TYPE': 'Service Type', 'TOTAL_AMOUNT': 'Amount (Ksh)'},
    text='AMOUNT_LABEL',               # Shows "75.6M" on top of bars
    color='TOTAL_AMOUNT',
    color_continuous_scale='Viridis'
)

# Customise bar labels and hover
fig.update_traces(
    textposition='outside',
    textfont_size=12,
    hovertemplate='%{customdata}<extra></extra>',
    customdata=service_summary['HOVER_TEXT']
)

# Improve layout
fig.update_layout(
    xaxis_title="Service Type",
    yaxis_title="Total Amount (Ksh)",
    height=500,
    showlegend=False,
    plot_bgcolor='white',
    xaxis=dict(gridcolor='lightgray'),
    yaxis=dict(gridcolor='lightgray')
)
fig.update_yaxes(tickformat=',.0f')   # correct plural method

# Show chart
fig.show()

# 3. Display a clean table (without Streamlit)
print("\n📊 Service Type Details Table")
display_df = service_summary[['SERVICE TYPE', 'TOTAL_AMOUNT', 'UNIQUE_VISITS']].copy()
display_df.columns = ['Service Type', 'Total Amount (Ksh)', 'Unique Visits']
display_df['Total Amount (Ksh)'] = display_df['Total Amount (Ksh)'].apply(lambda x: f'Ksh {x:,.0f}')
display_df['Unique Visits'] = display_df['Unique Visits'].apply(lambda x: f'{x:,}')

# For Jupyter, use display() for nicer output
from IPython.display import display
display(display_df)


📊 Service Type Details Table


,Service Type,Total Amount (Ksh),Unique Visits
0,IP,"Ksh 184,946,092","1,483"
1,OP,"Ksh 140,596,161","18,331"


In [111]:
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

# ==================================================
# BEAUTIFUL SERVICE TYPE CHART – CREATIVE & CLEAN
# ==================================================

# 1. Aggregate data
service = df.groupby('SERVICE TYPE').agg(
    TOTAL_AMOUNT=('AMOUNT', 'sum'),
    UNIQUE_VISITS=('VISIT_KEY', 'nunique')
).reset_index()

# 2. Calculate percentage
total_amount = service['TOTAL_AMOUNT'].sum()
service['PERCENT'] = (service['TOTAL_AMOUNT'] / total_amount) * 100

# 3. Format labels
service['AMOUNT_M'] = service['TOTAL_AMOUNT'] / 1e6
service['AMOUNT_LABEL'] = service['AMOUNT_M'].apply(lambda x: f'{x:.1f}M')
service['PERCENT_LABEL'] = service['PERCENT'].apply(lambda x: f'{x:.1f}%')
service['VISIT_LABEL'] = service['UNIQUE_VISITS'].apply(lambda x: f'👥 {x:,} visits')

# 4. Sort by amount (largest on top)
service = service.sort_values('TOTAL_AMOUNT', ascending=True)

# 5. Create horizontal bar chart
fig = go.Figure()

# Add bars – gradient blue
fig.add_trace(go.Bar(
    y=service['SERVICE TYPE'],
    x=service['TOTAL_AMOUNT'],
    orientation='h',
    marker=dict(
        color=service['TOTAL_AMOUNT'],
        colorscale='Blues',
        showscale=False,
        line=dict(color='darkblue', width=1)
    ),
    text=service['PERCENT_LABEL'],          # Show % inside bar
    textposition='inside',
    textfont=dict(color='white', size=14, family='Arial Black'),
    hovertemplate='<b>%{y}</b><br>' +
                  'Amount: %{x:,.0f} Ksh<br>' +
                  'Share: %{text}<br>' +
                  '%{customdata}<extra></extra>',
    customdata=service['VISIT_LABEL']
))

# 6. Add visit count as annotation outside the bar
for i, row in service.iterrows():
    fig.add_annotation(
        x=row['TOTAL_AMOUNT'] + (total_amount * 0.02),  # slight offset to the right
        y=row['SERVICE TYPE'],
        text=row['VISIT_LABEL'],
        showarrow=False,
        font=dict(size=12, color='#1f3b4c'),
        xanchor='left'
    )
    # Also add amount label at the very end (optional)
    fig.add_annotation(
        x=row['TOTAL_AMOUNT'] - (total_amount * 0.03),
        y=row['SERVICE TYPE'],
        text=row['AMOUNT_LABEL'],
        showarrow=False,
        font=dict(size=13, color='white', family='Arial Black'),
        xanchor='right'
    )

# 7. Layout styling
fig.update_layout(
    title=dict(
        text='💰 Service Type Breakdown: Amount, Share & Unique Visits',
        font=dict(size=20, color='#0a2b3e'),
        x=0.5
    ),
    xaxis=dict(
        title='Total Amount (Ksh)',
        tickformat=',.0f',
        gridcolor='lightgrey',
        showgrid=True,
        zeroline=False
    ),
    yaxis=dict(
        title='',
        categoryorder='total ascending',   # already sorted
        gridcolor='lightgrey'
    ),
    plot_bgcolor='#f8f9fa',
    height=400,
    margin=dict(l=100, r=120, t=80, b=50),
    hovermode='y unified'
)

fig.show()

# 8. Beautiful table with modern styling
print("\n📊 Service Type Summary (formatted)\n")
styled_table = service[['SERVICE TYPE', 'TOTAL_AMOUNT', 'UNIQUE_VISITS', 'PERCENT']].copy()
styled_table.columns = ['Service Type', 'Amount (Ksh)', 'Unique Visits', 'Share']
styled_table['Amount (Ksh)'] = styled_table['Amount (Ksh)'].apply(lambda x: f'Ksh {x:,.0f}')
styled_table['Unique Visits'] = styled_table['Unique Visits'].apply(lambda x: f'{x:,}')
styled_table['Share'] = styled_table['Share'].apply(lambda x: f'{x:.1f}%')

# Use display with pandas styling for zebra stripes & highlight
styled = styled_table.style.set_properties(**{'text-align': 'left'}) \
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#0a2b3e'), ('color', 'white'), ('font-weight', 'bold')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f0f4f8')]},
        {'selector': 'td', 'props': [('padding', '8px')]}
    ]) \
    .format({'Amount (Ksh)': lambda x: x, 'Unique Visits': lambda x: x, 'Share': lambda x: x})

display(styled)


📊 Service Type Summary (formatted)



,Service Type,Amount (Ksh),Unique Visits,Share
1,OP,"Ksh 140,596,161","18,331",43.2%
0,IP,"Ksh 184,946,092","1,483",56.8%


In [112]:
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

# ==================================================
# SERVICE TYPE CHART – MODERN THEMES (pick one)
# ==================================================

# 1. Aggregate data
service = df.groupby('SERVICE TYPE').agg(
    TOTAL_AMOUNT=('AMOUNT', 'sum'),
    UNIQUE_VISITS=('VISIT_KEY', 'nunique')
).reset_index()

# 2. Calculate percentage
total_amount = service['TOTAL_AMOUNT'].sum()
service['PERCENT'] = (service['TOTAL_AMOUNT'] / total_amount) * 100

# 3. Format labels
service['AMOUNT_M'] = service['TOTAL_AMOUNT'] / 1e6
service['AMOUNT_LABEL'] = service['AMOUNT_M'].apply(lambda x: f'{x:.1f}M')
service['PERCENT_LABEL'] = service['PERCENT'].apply(lambda x: f'{x:.1f}%')
service['VISIT_LABEL'] = service['UNIQUE_VISITS'].apply(lambda x: f'👥 {x:,} visits')

# Sort by amount (largest on top)
service = service.sort_values('TOTAL_AMOUNT', ascending=True)

# ==================================================
# CHOOSE YOUR THEME (uncomment one block)
# ==================================================

# THEME 1: Teal & Coral 🌿
# colors = ['#2c7a7a', '#3b9e9e', '#4fb3b3', '#80cbc4', '#a5d6d6']
# bar_color = 'teal'  # for gradient

# THEME 2: Sunset Gradient 🌅
colors = ['#ff6b6b', '#feca57', '#ff9ff3', '#ff6b6b', '#ee5a24']
bar_color = 'sunset'

# THEME 3: Forest & Mint 🍃
# colors = ['#2d6a4f', '#40916c', '#52b788', '#74c69d', '#95d5b2']
# bar_color = 'green'

# 4. Create bar chart with chosen theme
fig = go.Figure()

# Use gradient based on amount (linear colour scale)
fig.add_trace(go.Bar(
    y=service['SERVICE TYPE'],
    x=service['TOTAL_AMOUNT'],
    orientation='h',
    marker=dict(
        color=service['TOTAL_AMOUNT'],
        colorscale='Sunset' if bar_color == 'sunset' else ('Tealgrn' if bar_color == 'teal' else 'Greens'),
        showscale=False,
        line=dict(color='white', width=1.5)
    ),
    text=service['PERCENT_LABEL'],
    textposition='inside',
    textfont=dict(color='white', size=14, family='Arial Black'),
    hovertemplate='<b>%{y}</b><br>' +
                  '💰 Amount: %{x:,.0f} Ksh<br>' +
                  '📊 Share: %{text}<br>' +
                  '%{customdata}<extra></extra>',
    customdata=service['VISIT_LABEL']
))

# 5. Add visit count annotation (outside)
for i, row in service.iterrows():
    fig.add_annotation(
        x=row['TOTAL_AMOUNT'] + (total_amount * 0.02),
        y=row['SERVICE TYPE'],
        text=row['VISIT_LABEL'],
        showarrow=False,
        font=dict(size=12, color='#2c3e50', family='Arial'),
        xanchor='left'
    )
    # Add amount label inside bar (right side)
    fig.add_annotation(
        x=row['TOTAL_AMOUNT'] - (total_amount * 0.03),
        y=row['SERVICE TYPE'],
        text=row['AMOUNT_LABEL'],
        showarrow=False,
        font=dict(size=13, color='white', family='Arial Black'),
        xanchor='right'
    )

# 6. Modern layout using Plotly's built-in template
fig.update_layout(
    template='plotly_white',   # clean, grid only, no background colour
    title=dict(
        text='💰 Service Type Breakdown: Amount, Share & Unique Visits',
        font=dict(size=22, color='#1e2a3a', family='Arial Black'),
        x=0.5,
        xanchor='center'
    ),
    xaxis=dict(
        title='Total Amount (Ksh)',
        tickformat=',.0f',
        gridcolor='#e0e0e0',
        showgrid=True,
        zeroline=False,
        title_font=dict(size=13)
    ),
    yaxis=dict(
        title='',
        categoryorder='total ascending',
        gridcolor='#e0e0e0',
        tickfont=dict(size=12)
    ),
    plot_bgcolor='#fefefe',
    paper_bgcolor='#fefefe',
    height=450,
    margin=dict(l=120, r=140, t=80, b=50),
    hovermode='y unified'
)

fig.show()

# ==================================================
# BEAUTIFUL TABLE WITH MODERN STYLING
# ==================================================
print("\n📊 Service Type Summary\n")

styled_table = service[['SERVICE TYPE', 'TOTAL_AMOUNT', 'UNIQUE_VISITS', 'PERCENT']].copy()
styled_table.columns = ['Service Type', 'Amount (Ksh)', 'Unique Visits', 'Share (%)']
styled_table['Amount (Ksh)'] = styled_table['Amount (Ksh)'].apply(lambda x: f'Ksh {x:,.0f}')
styled_table['Unique Visits'] = styled_table['Unique Visits'].apply(lambda x: f'{x:,}')
styled_table['Share (%)'] = styled_table['Share (%)'].apply(lambda x: f'{x:.1f}%')

# Use seaborn-like styling
styled = styled_table.style \
    .set_properties(**{'text-align': 'left', 'border-collapse': 'collapse'}) \
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold'), ('padding', '8px')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f5f7fa')]},
        {'selector': 'td', 'props': [('padding', '8px'), ('border-bottom', '1px solid #ddd')]},
        {'selector': 'table', 'props': [('border', 'none')]}
    ]) \
    .hide(axis='index') \
    .set_caption("Data source: claims analysis")

display(styled)


📊 Service Type Summary



Service Type,Amount (Ksh),Unique Visits,Share (%)
OP,"Ksh 140,596,161","18,331",43.2%
IP,"Ksh 184,946,092","1,483",56.8%


In [113]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# ==================================================
# PATIENT HOSPITAL SWITCHING WITHIN 7 DAYS
# ==================================================

# 1. Prepare data: sort by member and arrival date
df_switches = df.copy()
df_switches = df_switches.sort_values(['MEMBER NUMBER', 'ARRIVAL DATE'])

# 2. Identify consecutive claims per member that are at different hospitals within 7 days
switches = []

for member, group in df_switches.groupby('MEMBER NUMBER'):
    if len(group) < 2:
        continue
    prev_row = None
    for idx, row in group.iterrows():
        if prev_row is not None:
            # Check if different hospital
            if row['MAIN HOSPITAL'] != prev_row['MAIN HOSPITAL']:
                date_diff = (row['ARRIVAL DATE'] - prev_row['ARRIVAL DATE']).days
                if 0 < date_diff <= 7:
                    switches.append({
                        'MEMBER NUMBER': member,
                        'HOSPITAL_FROM': prev_row['MAIN HOSPITAL'],
                        'HOSPITAL_TO': row['MAIN HOSPITAL'],
                        'DAYS_BETWEEN': date_diff,
                        'AMOUNT_FROM': prev_row['AMOUNT'],
                        'AMOUNT_TO': row['AMOUNT']
                    })
        prev_row = row

switches_df = pd.DataFrame(switches)

if switches_df.empty:
    print("No switching events found within 7 days.")
else:
    # 3. Aggregate per origin hospital
    origin_summary = switches_df.groupby('HOSPITAL_FROM').agg(
        SWITCHER_COUNT=('MEMBER NUMBER', 'nunique'),
        TOTAL_SWITCH_EVENTS=('MEMBER NUMBER', 'count'),
        AVG_DAYS=('DAYS_BETWEEN', 'mean')
    ).reset_index()

    # 4. Total unique patients per hospital (any visit)
    total_patients = df.groupby('MAIN HOSPITAL')['MEMBER NUMBER'].nunique().reset_index()
    total_patients.columns = ['HOSPITAL_FROM', 'TOTAL_PATIENTS']

    # 5. Merge and compute switch-out rate
    hospital_ranking = origin_summary.merge(total_patients, on='HOSPITAL_FROM', how='right').fillna(0)
    hospital_ranking['SWITCH_OUT_RATE'] = (hospital_ranking['SWITCHER_COUNT'] / hospital_ranking['TOTAL_PATIENTS']) * 100
    hospital_ranking = hospital_ranking.sort_values('SWITCH_OUT_RATE', ascending=False)

    # 6. Display worst hospitals
    print("🏥 HOSPITAL RANKING – WORST RETENTION (highest % of patients leaving within 7 days)\n")
    display_rank = hospital_ranking[['HOSPITAL_FROM', 'TOTAL_PATIENTS', 'SWITCHER_COUNT', 'SWITCH_OUT_RATE', 'AVG_DAYS']].copy()
    display_rank.columns = ['Hospital', 'Total Patients', 'Patients Who Left', 'Switch-out Rate (%)', 'Avg Days to Switch']
    display_rank['Switch-out Rate (%)'] = display_rank['Switch-out Rate (%)'].round(1)
    display_rank['Avg Days to Switch'] = display_rank['Avg Days to Switch'].round(1)
    display(display_rank.head(10))  # top 10 worst

    # 7. Bar chart: worst hospitals (top 10)
    top_worst = hospital_ranking.head(10)
    fig = px.bar(
        top_worst,
        x='SWITCH_OUT_RATE',
        y='HOSPITAL_FROM',
        orientation='h',
        title='📉 Hospitals with Highest Patient Defection (within 7 days)',
        labels={'SWITCH_OUT_RATE': 'Switch-out Rate (%)', 'HOSPITAL_FROM': ''},
        text='SWITCH_OUT_RATE',
        color='SWITCH_OUT_RATE',
        color_continuous_scale='Reds'
    )
    fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    fig.update_layout(height=500, margin=dict(l=150))
    fig.show()

    # 8. Optional: Sankey diagram of flows (top 5 origins to top 5 destinations)
    top_origins = hospital_ranking.head(5)['HOSPITAL_FROM'].tolist()
    flow_df = switches_df[switches_df['HOSPITAL_FROM'].isin(top_origins)]
    flow_sum = flow_df.groupby(['HOSPITAL_FROM', 'HOSPITAL_TO']).size().reset_index(name='count')
    flow_sum = flow_sum.sort_values('count', ascending=False).head(15)

    if not flow_sum.empty:
        # Create Sankey
        labels = list(set(flow_sum['HOSPITAL_FROM'].unique()) | set(flow_sum['HOSPITAL_TO'].unique()))
        label_to_index = {label: i for i, label in enumerate(labels)}
        source = [label_to_index[h] for h in flow_sum['HOSPITAL_FROM']]
        target = [label_to_index[h] for h in flow_sum['HOSPITAL_TO']]
        value = flow_sum['count']

        fig_sankey = go.Figure(data=[go.Sankey(
            node=dict(pad=15, thickness=20, line=dict(color='black', width=0.5), label=labels),
            link=dict(source=source, target=target, value=value)
        )])
        fig_sankey.update_layout(title='Patient Flows: Top 5 Worst Hospitals → Where They Go', height=600)
        fig_sankey.show()
    else:
        print("Not enough flow data for Sankey diagram.")

🏥 HOSPITAL RANKING – WORST RETENTION (highest % of patients leaving within 7 days)



,Hospital,Total Patients,Patients Who Left,Switch-out Rate (%),Avg Days to Switch
7,BESTCARE HOSPITAL LIMITED,3,2.0,66.7,4.0
94,SPINE CLINIC AFRICA LTD,5,2.0,40.0,2.8
13,CANCER CARE CENTRE,15,6.0,40.0,4.9
24,ELGON VIEW HOSP ELDORET,9,3.0,33.3,5.3
39,KILOME MATERNITY NURSING HOME,6,2.0,33.3,3.5
122,THE MITUNGUU HOSP LTD,3,1.0,33.3,4.0
37,KENYATTA NATIONAL HOSP,15,4.0,26.7,2.2
38,KENYATTA UNIVERSITY HOSPITAL (KUTRR),23,6.0,26.1,2.4
109,ST MONICAH MISSION,4,1.0,25.0,5.0
111,STARKEY HEARING TECHNOLOGIES LTD,8,2.0,25.0,5.0


In [114]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# ==================================================
# PATIENT HOSPITAL SWITCHING WITHIN 7 DAYS
# RANK BY NUMBER OF PATIENTS WHO LEFT (absolute)
# ==================================================

# 1. Prepare data: sort by member and arrival date
df_switches = df.copy()
df_switches = df_switches.sort_values(['MEMBER NUMBER', 'ARRIVAL DATE'])

# 2. Identify consecutive claims per member at different hospitals within 7 days
switches = []

for member, group in df_switches.groupby('MEMBER NUMBER'):
    if len(group) < 2:
        continue
    prev_row = None
    for idx, row in group.iterrows():
        if prev_row is not None:
            if row['MAIN HOSPITAL'] != prev_row['MAIN HOSPITAL']:
                date_diff = (row['ARRIVAL DATE'] - prev_row['ARRIVAL DATE']).days
                if 0 < date_diff <= 7:
                    switches.append({
                        'MEMBER NUMBER': member,
                        'HOSPITAL_FROM': prev_row['MAIN HOSPITAL'],
                        'HOSPITAL_TO': row['MAIN HOSPITAL'],
                        'DAYS_BETWEEN': date_diff,
                        'AMOUNT_FROM': prev_row['AMOUNT'],
                        'AMOUNT_TO': row['AMOUNT']
                    })
        prev_row = row

switches_df = pd.DataFrame(switches)

if switches_df.empty:
    print("No switching events found within 7 days.")
else:
    # 3. Aggregate per origin hospital
    origin_summary = switches_df.groupby('HOSPITAL_FROM').agg(
        PATIENTS_LEFT=('MEMBER NUMBER', 'nunique'),
        TOTAL_SWITCH_EVENTS=('MEMBER NUMBER', 'count'),
        AVG_DAYS=('DAYS_BETWEEN', 'mean')
    ).reset_index()

    # 4. Total unique patients per hospital (any visit)
    total_patients = df.groupby('MAIN HOSPITAL')['MEMBER NUMBER'].nunique().reset_index()
    total_patients.columns = ['HOSPITAL_FROM', 'TOTAL_PATIENTS']

    # 5. Merge and compute switch-out rate (keep as optional)
    hospital_ranking = origin_summary.merge(total_patients, on='HOSPITAL_FROM', how='right').fillna(0)
    hospital_ranking['SWITCH_OUT_RATE'] = (hospital_ranking['PATIENTS_LEFT'] / hospital_ranking['TOTAL_PATIENTS']) * 100
    # Rank by absolute number of patients who left (descending – worst first)
    hospital_ranking = hospital_ranking.sort_values('PATIENTS_LEFT', ascending=False)

    # 6. Display worst hospitals by absolute patient loss
    print("🏥 HOSPITAL RANKING – WORST (most patients who left within 7 days)\n")
    display_rank = hospital_ranking[['HOSPITAL_FROM', 'TOTAL_PATIENTS', 'PATIENTS_LEFT', 'SWITCH_OUT_RATE', 'AVG_DAYS']].copy()
    display_rank.columns = ['Hospital', 'Total Patients', 'Patients Who Left', 'Switch-out Rate (%)', 'Avg Days to Switch']
    display_rank['Patients Who Left'] = display_rank['Patients Who Left'].astype(int)
    display_rank['Switch-out Rate (%)'] = display_rank['Switch-out Rate (%)'].round(1)
    display_rank['Avg Days to Switch'] = display_rank['Avg Days to Switch'].round(1)
    display(display_rank.head(10))

    # 7. Bar chart: worst hospitals by number of patients left (top 10)
    top_worst = hospital_ranking.head(10)
    fig = px.bar(
        top_worst,
        x='PATIENTS_LEFT',
        y='HOSPITAL_FROM',
        orientation='h',
        title='📉 Hospitals with Highest Patient Defection (absolute count, within 7 days)',
        labels={'PATIENTS_LEFT': 'Number of Patients Who Left', 'HOSPITAL_FROM': ''},
        text='PATIENTS_LEFT',
        color='PATIENTS_LEFT',
        color_continuous_scale='Reds'
    )
    fig.update_traces(texttemplate='%{text}', textposition='outside')
    fig.update_layout(height=500, margin=dict(l=150))
    fig.show()

    # 8. Sankey diagram of flows (top 5 origins by patient loss)
    top_origins = hospital_ranking.head(5)['HOSPITAL_FROM'].tolist()
    flow_df = switches_df[switches_df['HOSPITAL_FROM'].isin(top_origins)]
    flow_sum = flow_df.groupby(['HOSPITAL_FROM', 'HOSPITAL_TO']).size().reset_index(name='count')
    flow_sum = flow_sum.sort_values('count', ascending=False).head(15)

    if not flow_sum.empty:
        labels = list(set(flow_sum['HOSPITAL_FROM'].unique()) | set(flow_sum['HOSPITAL_TO'].unique()))
        label_to_index = {label: i for i, label in enumerate(labels)}
        source = [label_to_index[h] for h in flow_sum['HOSPITAL_FROM']]
        target = [label_to_index[h] for h in flow_sum['HOSPITAL_TO']]
        value = flow_sum['count']

        fig_sankey = go.Figure(data=[go.Sankey(
            node=dict(pad=15, thickness=20, line=dict(color='black', width=0.5), label=labels),
            link=dict(source=source, target=target, value=value)
        )])
        fig_sankey.update_layout(title='Patient Flows: Top 5 Worst Hospitals → Where They Go', height=600)
        fig_sankey.show()
    else:
        print("Not enough flow data for Sankey diagram.")

🏥 HOSPITAL RANKING – WORST (most patients who left within 7 days)



,Hospital,Total Patients,Patients Who Left,Switch-out Rate (%),Avg Days to Switch
128,ULINZI PRIME HEALTH SERVICES FUND (UPHSF),1982,205,10.3,3.2
26,EQUITY AFIA RONGAI,834,77,9.2,3.4
56,MONALIFE PHARMACEUTICALS LTD,401,43,10.7,3.6
64,NAIROBI WEST HOSP,230,38,16.5,3.7
0,AAR HEALTHCARE NAIROBI,275,34,12.4,3.8
120,THE KAREN HOSP REFERRAL,170,34,20.0,3.6
62,NAIROBI HOSP REFERRAL,195,27,13.8,3.8
9,BLOOM HOSPITAL NAKURU LIMITED,120,15,12.5,4.5
72,OPTICA LIMITED,164,15,9.1,3.9
99,ST FRANCIS COMMUNITY KASARANI,151,15,9.9,3.9


In [115]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# ==================================================
# PATIENT COMPLETE SWITCH: FIRST HOSPITAL → LAST HOSPITAL
# ==================================================

# 1. Group by member, get first and last hospital
member_first_last = df.groupby('MEMBER NUMBER').agg(
    FIRST_DATE=('ARRIVAL DATE', 'min'),
    LAST_DATE=('ARRIVAL DATE', 'max'),
    FIRST_HOSPITAL=('MAIN HOSPITAL', 'first'),   # first occurrence in original order? better: use min date
    LAST_HOSPITAL=('MAIN HOSPITAL', 'last'),
    VISIT_COUNT=('CLAIM ID', 'count')
).reset_index()

# But 'first' and 'last' may not align with min/max dates if data not sorted. Better:
# Get hospital associated with min date and max date
def get_first_hospital(group):
    return group.loc[group['ARRIVAL DATE'].idxmin(), 'MAIN HOSPITAL']

def get_last_hospital(group):
    return group.loc[group['ARRIVAL DATE'].idxmax(), 'MAIN HOSPITAL']

member_first_hospital = df.groupby('MEMBER NUMBER').apply(get_first_hospital).reset_index(name='FIRST_HOSPITAL')
member_last_hospital = df.groupby('MEMBER NUMBER').apply(get_last_hospital).reset_index(name='LAST_HOSPITAL')
member_visits = df.groupby('MEMBER NUMBER')['CLAIM ID'].count().reset_index(name='VISIT_COUNT')
member_dates = df.groupby('MEMBER NUMBER')['ARRIVAL DATE'].agg(['min', 'max']).reset_index()
member_dates.columns = ['MEMBER NUMBER', 'FIRST_DATE', 'LAST_DATE']

# Merge
member_summary = member_first_hospital.merge(member_last_hospital, on='MEMBER NUMBER')
member_summary = member_summary.merge(member_visits, on='MEMBER NUMBER')
member_summary = member_summary.merge(member_dates, on='MEMBER NUMBER')
member_summary['DATE_RANGE_DAYS'] = (member_summary['LAST_DATE'] - member_summary['FIRST_DATE']).dt.days

# Flag switchers (first ≠ last)
member_summary['SWITCHED'] = member_summary['FIRST_HOSPITAL'] != member_summary['LAST_HOSPITAL']

# 2. Aggregate by first hospital
hospital_switch = member_summary.groupby('FIRST_HOSPITAL').agg(
    PATIENTS_STARTED=('MEMBER NUMBER', 'nunique'),
    PATIENTS_LEFT=('SWITCHED', 'sum'),          # count where SWITCHED is True
    TOTAL_VISITS=('VISIT_COUNT', 'sum'),
    AVG_VISITS=('VISIT_COUNT', 'mean'),
    AVG_DATE_RANGE=('DATE_RANGE_DAYS', 'mean')
).reset_index()

hospital_switch['RETENTION_RATE'] = (1 - hospital_switch['PATIENTS_LEFT'] / hospital_switch['PATIENTS_STARTED']) * 100
hospital_switch = hospital_switch.sort_values('PATIENTS_LEFT', ascending=False)

# 3. Display worst hospitals (most patients who left)
print("🏥 HOSPITAL RANKING – WORST (most patients who started there but ended elsewhere)\n")
display_cols = ['FIRST_HOSPITAL', 'PATIENTS_STARTED', 'PATIENTS_LEFT', 'RETENTION_RATE', 'AVG_VISITS', 'AVG_DATE_RANGE']
display_rank = hospital_switch[display_cols].copy()
display_rank.columns = ['Hospital', 'Patients Started', 'Patients Who Left', 'Retention (%)', 'Avg Visits per Patient', 'Avg Days from First to Last']
display_rank['Retention (%)'] = display_rank['Retention (%)'].round(1)
display_rank['Avg Visits per Patient'] = display_rank['Avg Visits per Patient'].round(1)
display_rank['Avg Days from First to Last'] = display_rank['Avg Days from First to Last'].round(0).astype(int)
display(display_rank.head(10))

# 4. Bar chart – worst by absolute patient loss
top_worst = hospital_switch.head(10)
fig = px.bar(
    top_worst,
    x='PATIENTS_LEFT',
    y='FIRST_HOSPITAL',
    orientation='h',
    title='📉 Hospitals with Most Patient Defections (complete switch, first ≠ last hospital)',
    labels={'PATIENTS_LEFT': 'Number of Patients Who Left', 'FIRST_HOSPITAL': ''},
    text='PATIENTS_LEFT',
    color='PATIENTS_LEFT',
    color_continuous_scale='Reds'
)
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(height=500, margin=dict(l=150))
fig.show()

# 5. Sankey diagram: flow from first hospital to last hospital (top origins)
top_origins = hospital_switch.head(5)['FIRST_HOSPITAL'].tolist()
flow_data = member_summary[member_summary['FIRST_HOSPITAL'].isin(top_origins) & member_summary['SWITCHED']]
flow_sum = flow_data.groupby(['FIRST_HOSPITAL', 'LAST_HOSPITAL']).size().reset_index(name='count')
flow_sum = flow_sum.sort_values('count', ascending=False).head(20)

if not flow_sum.empty:
    labels = list(set(flow_sum['FIRST_HOSPITAL'].unique()) | set(flow_sum['LAST_HOSPITAL'].unique()))
    label_to_index = {label: i for i, label in enumerate(labels)}
    source = [label_to_index[h] for h in flow_sum['FIRST_HOSPITAL']]
    target = [label_to_index[h] for h in flow_sum['LAST_HOSPITAL']]
    value = flow_sum['count']

    fig_sankey = go.Figure(data=[go.Sankey(
        node=dict(pad=15, thickness=20, line=dict(color='black', width=0.5), label=labels),
        link=dict(source=source, target=target, value=value)
    )])
    fig_sankey.update_layout(title='Patient Flow: First Hospital → Last Hospital (Top 5 Origins)', height=600)
    fig_sankey.show()
else:
    print("Not enough flow data for Sankey diagram.")

# 6. Optional: Show patients who never switched (loyal) vs switchers
member_summary['SWITCH_STATUS'] = member_summary['SWITCHED'].map({True: 'Switched', False: 'Loyal'})
status_counts = member_summary['SWITCH_STATUS'].value_counts()
fig_pie = px.pie(values=status_counts.values, names=status_counts.index, title='Patient Loyalty vs Switching')
fig_pie.show()

🏥 HOSPITAL RANKING – WORST (most patients who started there but ended elsewhere)



,Hospital,Patients Started,Patients Who Left,Retention (%),Avg Visits per Patient,Avg Days from First to Last
125,ULINZI PRIME HEALTH SERVICES FUND (UPHSF),1656,259,84.4,3.2,37
26,EQUITY AFIA RONGAI,662,148,77.6,2.8,34
0,AAR HEALTHCARE NAIROBI,199,48,75.9,4.0,45
62,NAIROBI WEST HOSP,150,46,69.3,6.9,53
60,NAIROBI HOSP REFERRAL,145,32,77.9,5.5,57
117,THE KAREN HOSP REFERRAL,117,31,73.5,4.4,48
73,PANDYA MEMORIAL HOSP,86,27,68.6,5.5,39
55,MONALIFE PHARMACEUTICALS LTD,82,23,72.0,2.6,38
114,THE AGA KHAN HOSP KIS,158,22,86.1,5.7,45
56,MOTHER ANGELA HURUMA HOSPITAL,99,22,77.8,3.3,46


In [116]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# ==================================================
# BEST & WORST HOSPITALS BY ABSOLUTE PATIENT RETENTION
# (Complete switch: first hospital ≠ last hospital)
# ==================================================

# 1. Helper to get hospital at min/max date
def get_first_hospital(group):
    return group.loc[group['ARRIVAL DATE'].idxmin(), 'MAIN HOSPITAL']

def get_last_hospital(group):
    return group.loc[group['ARRIVAL DATE'].idxmax(), 'MAIN HOSPITAL']

member_first = df.groupby('MEMBER NUMBER').apply(get_first_hospital).reset_index(name='FIRST_HOSPITAL')
member_last = df.groupby('MEMBER NUMBER').apply(get_last_hospital).reset_index(name='LAST_HOSPITAL')
member_visits = df.groupby('MEMBER NUMBER')['CLAIM ID'].count().reset_index(name='VISIT_COUNT')
member_dates = df.groupby('MEMBER NUMBER')['ARRIVAL DATE'].agg(['min', 'max']).reset_index()
member_dates.columns = ['MEMBER NUMBER', 'FIRST_DATE', 'LAST_DATE']

member_summary = member_first.merge(member_last, on='MEMBER NUMBER')
member_summary = member_summary.merge(member_visits, on='MEMBER NUMBER')
member_summary = member_summary.merge(member_dates, on='MEMBER NUMBER')
member_summary['DATE_RANGE_DAYS'] = (member_summary['LAST_DATE'] - member_summary['FIRST_DATE']).dt.days
member_summary['SWITCHED'] = member_summary['FIRST_HOSPITAL'] != member_summary['LAST_HOSPITAL']

# 2. Aggregate by first hospital
hospital_stats = member_summary.groupby('FIRST_HOSPITAL').agg(
    PATIENTS_STARTED=('MEMBER NUMBER', 'nunique'),
    PATIENTS_LEFT=('SWITCHED', 'sum')
).reset_index()
hospital_stats['PATIENTS_RETAINED'] = hospital_stats['PATIENTS_STARTED'] - hospital_stats['PATIENTS_LEFT']
hospital_stats['RETENTION_RATE'] = (hospital_stats['PATIENTS_RETAINED'] / hospital_stats['PATIENTS_STARTED']) * 100

# 3. Best hospitals by absolute number retained (descending)
best_hospitals = hospital_stats.sort_values('PATIENTS_RETAINED', ascending=False)
worst_hospitals = hospital_stats.sort_values('PATIENTS_LEFT', ascending=False)

# 4. Display tables
print("🏆 BEST HOSPITALS – Most patients retained (absolute count)\n")
best_display = best_hospitals[['FIRST_HOSPITAL', 'PATIENTS_STARTED', 'PATIENTS_RETAINED', 'RETENTION_RATE']].head(10).copy()
best_display.columns = ['Hospital', 'Patients Started', 'Patients Retained', 'Retention Rate (%)']
best_display['Retention Rate (%)'] = best_display['Retention Rate (%)'].round(1)
display(best_display)

print("\n📉 WORST HOSPITALS – Most patients lost (absolute count)\n")
worst_display = worst_hospitals[['FIRST_HOSPITAL', 'PATIENTS_STARTED', 'PATIENTS_LEFT', 'RETENTION_RATE']].head(10).copy()
worst_display.columns = ['Hospital', 'Patients Started', 'Patients Lost', 'Retention Rate (%)']
worst_display['Retention Rate (%)'] = worst_display['Retention Rate (%)'].round(1)
display(worst_display)

# 5. Bar chart: Best by retained count
fig_best = px.bar(
    best_hospitals.head(10),
    x='PATIENTS_RETAINED',
    y='FIRST_HOSPITAL',
    orientation='h',
    title='🏥 Best Hospitals: Highest Number of Patients Retained (first → same hospital)',
    labels={'PATIENTS_RETAINED': 'Number of Patients Retained', 'FIRST_HOSPITAL': ''},
    text='PATIENTS_RETAINED',
    color='PATIENTS_RETAINED',
    color_continuous_scale='Greens'
)
fig_best.update_traces(texttemplate='%{text}', textposition='outside')
fig_best.update_layout(height=500, margin=dict(l=150))
fig_best.show()

# 6. Bar chart: Worst by lost count
fig_worst = px.bar(
    worst_hospitals.head(10),
    x='PATIENTS_LEFT',
    y='FIRST_HOSPITAL',
    orientation='h',
    title='📉 Worst Hospitals: Highest Number of Patients Lost (first → different hospital)',
    labels={'PATIENTS_LEFT': 'Number of Patients Lost', 'FIRST_HOSPITAL': ''},
    text='PATIENTS_LEFT',
    color='PATIENTS_LEFT',
    color_continuous_scale='Reds'
)
fig_worst.update_traces(texttemplate='%{text}', textposition='outside')
fig_worst.update_layout(height=500, margin=dict(l=150))
fig_worst.show()

# 7. Optional: Sankey for worst hospitals (where they go)
top_worst_origins = worst_hospitals.head(5)['FIRST_HOSPITAL'].tolist()
flow_data = member_summary[
    member_summary['FIRST_HOSPITAL'].isin(top_worst_origins) & 
    member_summary['SWITCHED']
]
flow_sum = flow_data.groupby(['FIRST_HOSPITAL', 'LAST_HOSPITAL']).size().reset_index(name='count')
flow_sum = flow_sum.sort_values('count', ascending=False).head(15)

if not flow_sum.empty:
    labels = list(set(flow_sum['FIRST_HOSPITAL'].unique()) | set(flow_sum['LAST_HOSPITAL'].unique()))
    label_to_index = {label: i for i, label in enumerate(labels)}
    source = [label_to_index[h] for h in flow_sum['FIRST_HOSPITAL']]
    target = [label_to_index[h] for h in flow_sum['LAST_HOSPITAL']]
    value = flow_sum['count']

    fig_sankey = go.Figure(data=[go.Sankey(
        node=dict(pad=15, thickness=20, line=dict(color='black', width=0.5), label=labels),
        link=dict(source=source, target=target, value=value)
    )])
    fig_sankey.update_layout(title='Where Patients from Worst Hospitals Go (first → last hospital)', height=600)
    fig_sankey.show()
else:
    print("Not enough flow data for Sankey diagram.")

🏆 BEST HOSPITALS – Most patients retained (absolute count)



,Hospital,Patients Started,Patients Retained,Retention Rate (%)
125,ULINZI PRIME HEALTH SERVICES FUND (UPHSF),1656,1397,84.4
26,EQUITY AFIA RONGAI,662,514,77.6
0,AAR HEALTHCARE NAIROBI,199,151,75.9
114,THE AGA KHAN HOSP KIS,158,136,86.1
60,NAIROBI HOSP REFERRAL,145,113,77.9
62,NAIROBI WEST HOSP,150,104,69.3
96,ST FRANCIS COMMUNITY KASARANI,122,101,82.8
100,ST LUKE ORTHOPEADIC ELD,121,100,82.6
8,BISHOP KIOKO CATHOLIC,112,95,84.8
105,ST MATIA MULUMBA MISSION,100,89,89.0



📉 WORST HOSPITALS – Most patients lost (absolute count)



,Hospital,Patients Started,Patients Lost,Retention Rate (%)
125,ULINZI PRIME HEALTH SERVICES FUND (UPHSF),1656,259,84.4
26,EQUITY AFIA RONGAI,662,148,77.6
0,AAR HEALTHCARE NAIROBI,199,48,75.9
62,NAIROBI WEST HOSP,150,46,69.3
60,NAIROBI HOSP REFERRAL,145,32,77.9
117,THE KAREN HOSP REFERRAL,117,31,73.5
73,PANDYA MEMORIAL HOSP,86,27,68.6
55,MONALIFE PHARMACEUTICALS LTD,82,23,72.0
114,THE AGA KHAN HOSP KIS,158,22,86.1
56,MOTHER ANGELA HURUMA HOSPITAL,99,22,77.8


In [118]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22944 entries, 0 to 22943
Data columns (total 47 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   CLAIM ID                   22944 non-null  int64         
 1   CENTRAL ID                 22944 non-null  int64         
 2   CLAIM TYPE                 22944 non-null  object        
 3   SCHEME                     22944 non-null  object        
 4   MEMBER NUMBER              22944 non-null  object        
 5   INTEG MEMBER NUMBER        22943 non-null  float64       
 6   OTHER NUMBER               22944 non-null  object        
 7   OFFICE BRANCH              39 non-null     object        
 8   CARD SERIAL                22754 non-null  object        
 9   PATIENT NAME               22944 non-null  object        
 10  DOB                        22944 non-null  datetime64[ns]
 11  CAT CODE                   22944 non-null  object        
 12  CAT 

In [121]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# ==================================================
# PATIENT SWITCH ANALYSIS WITH POST-SWITCH VISITS
# Corrected column names: 'MAIN HOSPITAL' (space)
# ==================================================

# 1. Sort by member and date
df_sorted = df.sort_values(['MEMBER NUMBER', 'ARRIVAL DATE'])

# 2. For each member, get first hospital and first date
first_visit = df_sorted.groupby('MEMBER NUMBER').first().reset_index()[['MEMBER NUMBER', 'MAIN HOSPITAL', 'ARRIVAL DATE']]
first_visit.columns = ['MEMBER NUMBER', 'FIRST_HOSPITAL', 'FIRST_DATE']

# 3. Get all visits after first date that are at a different hospital
post_first = df_sorted.merge(first_visit, on='MEMBER NUMBER')
post_first = post_first[post_first['ARRIVAL DATE'] > post_first['FIRST_DATE']]
post_first = post_first[post_first['MAIN HOSPITAL'] != post_first['FIRST_HOSPITAL']]

# 4. Aggregate post-switch behaviour per member (correct column name)
member_post = post_first.groupby('MEMBER NUMBER').agg(
    UNIQUE_OTHER_HOSPITALS=('MAIN HOSPITAL', 'nunique'),   # now correct
    TOTAL_POST_VISITS=('CLAIM ID', 'count'),
    FIRST_POST_DATE=('ARRIVAL DATE', 'min'),
    LAST_POST_DATE=('ARRIVAL DATE', 'max')
).reset_index()

# 5. Merge with first visit info
member_switches = first_visit.merge(member_post, on='MEMBER NUMBER', how='left').fillna({
    'UNIQUE_OTHER_HOSPITALS': 0,
    'TOTAL_POST_VISITS': 0
})
member_switches['SWITCHED'] = member_switches['UNIQUE_OTHER_HOSPITALS'] > 0

# 6. Aggregate by FIRST_HOSPITAL
hospital_stats = member_switches.groupby('FIRST_HOSPITAL').agg(
    PATIENTS_STARTED=('MEMBER NUMBER', 'nunique'),
    PATIENTS_SWITCHED=('SWITCHED', 'sum'),
    TOTAL_UNIQUE_OTHER_HOSP_ALL=('UNIQUE_OTHER_HOSPITALS', 'sum'),
    TOTAL_POST_VISITS_ALL=('TOTAL_POST_VISITS', 'sum'),
    AVG_UNIQUE_OTHER_PER_SWITCHER=('UNIQUE_OTHER_HOSPITALS', 'mean'),
    AVG_POST_VISITS_PER_SWITCHER=('TOTAL_POST_VISITS', 'mean')
).reset_index()

hospital_stats['PATIENTS_RETAINED'] = hospital_stats['PATIENTS_STARTED'] - hospital_stats['PATIENTS_SWITCHED']
hospital_stats['RETENTION_RATE'] = (hospital_stats['PATIENTS_RETAINED'] / hospital_stats['PATIENTS_STARTED']) * 100

# 7. Rank worst by absolute number of patients who switched
worst = hospital_stats.sort_values('PATIENTS_SWITCHED', ascending=False)
best = hospital_stats.sort_values('PATIENTS_RETAINED', ascending=False)

# 8. Display tables
print("🏥 WORST HOSPITALS – Most patients who left and visited other facilities\n")
worst_display = worst[['FIRST_HOSPITAL', 'PATIENTS_STARTED', 'PATIENTS_SWITCHED', 
                       'AVG_UNIQUE_OTHER_PER_SWITCHER', 'AVG_POST_VISITS_PER_SWITCHER', 'RETENTION_RATE']].head(10).copy()
worst_display.columns = ['Hospital', 'Patients Started', 'Patients Switched', 
                         'Avg Unique Other Hospitals (per switcher)', 'Avg Post-Switch Visits (per switcher)', 'Retention Rate (%)']
worst_display['Retention Rate (%)'] = worst_display['Retention Rate (%)'].round(1)
worst_display['Avg Unique Other Hospitals (per switcher)'] = worst_display['Avg Unique Other Hospitals (per switcher)'].round(1)
worst_display['Avg Post-Switch Visits (per switcher)'] = worst_display['Avg Post-Switch Visits (per switcher)'].round(1)
display(worst_display)

print("\n🏆 BEST HOSPITALS – Most patients retained (never switched)\n")
best_display = best[['FIRST_HOSPITAL', 'PATIENTS_STARTED', 'PATIENTS_RETAINED', 'RETENTION_RATE']].head(10).copy()
best_display.columns = ['Hospital', 'Patients Started', 'Patients Retained', 'Retention Rate (%)']
best_display['Retention Rate (%)'] = best_display['Retention Rate (%)'].round(1)
display(best_display)

# 9. Bar chart – worst by switcher count
fig_worst = px.bar(
    worst.head(10),
    x='PATIENTS_SWITCHED',
    y='FIRST_HOSPITAL',
    orientation='h',
    title='📉 Hospitals with Most Patients Who Switched to Other Facilities',
    labels={'PATIENTS_SWITCHED': 'Number of Patients Who Switched', 'FIRST_HOSPITAL': ''},
    text='PATIENTS_SWITCHED',
    color='PATIENTS_SWITCHED',
    color_continuous_scale='Reds'
)
fig_worst.update_traces(texttemplate='%{text}', textposition='outside')
fig_worst.update_layout(height=500, margin=dict(l=150))
fig_worst.show()

# 10. Scatter plot: switchers vs average unique other hospitals visited
fig_scatter = px.scatter(
    hospital_stats,
    x='PATIENTS_SWITCHED',
    y='AVG_UNIQUE_OTHER_PER_SWITCHER',
    size='PATIENTS_STARTED',
    hover_name='FIRST_HOSPITAL',
    title='Patient Switching Volume vs Hospital Hopping (per switcher)',
    labels={'PATIENTS_SWITCHED': 'Number of Patients Who Switched', 
            'AVG_UNIQUE_OTHER_PER_SWITCHER': 'Avg Unique Other Hospitals per Switcher'}
)
fig_scatter.show()

🏥 WORST HOSPITALS – Most patients who left and visited other facilities



,Hospital,Patients Started,Patients Switched,Avg Unique Other Hospitals (per switcher),Avg Post-Switch Visits (per switcher),Retention Rate (%)
125,ULINZI PRIME HEALTH SERVICES FUND (UPHSF),1656,421,0.3,0.6,74.6
26,EQUITY AFIA RONGAI,662,188,0.4,0.8,71.6
0,AAR HEALTHCARE NAIROBI,199,65,0.5,1.1,67.3
62,NAIROBI WEST HOSP,150,59,0.5,1.3,60.7
60,NAIROBI HOSP REFERRAL,145,50,0.5,1.2,65.5
117,THE KAREN HOSP REFERRAL,117,46,0.5,1.0,60.7
73,PANDYA MEMORIAL HOSP,86,38,0.6,1.2,55.8
55,MONALIFE PHARMACEUTICALS LTD,82,36,0.6,1.2,56.1
114,THE AGA KHAN HOSP KIS,158,35,0.3,0.8,77.8
96,ST FRANCIS COMMUNITY KASARANI,122,33,0.3,0.7,73.0



🏆 BEST HOSPITALS – Most patients retained (never switched)



,Hospital,Patients Started,Patients Retained,Retention Rate (%)
125,ULINZI PRIME HEALTH SERVICES FUND (UPHSF),1656,1235,74.6
26,EQUITY AFIA RONGAI,662,474,71.6
0,AAR HEALTHCARE NAIROBI,199,134,67.3
114,THE AGA KHAN HOSP KIS,158,123,77.8
60,NAIROBI HOSP REFERRAL,145,95,65.5
100,ST LUKE ORTHOPEADIC ELD,121,94,77.7
62,NAIROBI WEST HOSP,150,91,60.7
96,ST FRANCIS COMMUNITY KASARANI,122,89,73.0
8,BISHOP KIOKO CATHOLIC,112,88,78.6
105,ST MATIA MULUMBA MISSION,100,85,85.0


In [122]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# ==================================================
# PATIENT SWITCH ANALYSIS – ACTUAL NUMBERS (NO AVERAGES)
# Shows total post‑switch visits per hospital and per patient
# ==================================================

# 1. Sort by member and date
df_sorted = df.sort_values(['MEMBER NUMBER', 'ARRIVAL DATE'])

# 2. For each member, get first hospital and first date
first_visit = df_sorted.groupby('MEMBER NUMBER').first().reset_index()[['MEMBER NUMBER', 'MAIN HOSPITAL', 'ARRIVAL DATE']]
first_visit.columns = ['MEMBER NUMBER', 'FIRST_HOSPITAL', 'FIRST_DATE']

# 3. Get all visits after first date that are at a different hospital
post_first = df_sorted.merge(first_visit, on='MEMBER NUMBER')
post_first = post_first[post_first['ARRIVAL DATE'] > post_first['FIRST_DATE']]
post_first = post_first[post_first['MAIN HOSPITAL'] != post_first['FIRST_HOSPITAL']]

# 4. Per patient: count of unique visits (i.e., number of rows) after switch
#    Also count unique other hospitals visited
member_post = post_first.groupby('MEMBER NUMBER').agg(
    UNIQUE_OTHER_HOSPITALS=('MAIN HOSPITAL', 'nunique'),   # distinct hospitals visited after switch
    TOTAL_POST_VISITS=('CLAIM ID', 'count')                # actual number of claim visits after switch
).reset_index()

# 5. Merge with first hospital info, fill non‑switchers with 0
member_switches = first_visit.merge(member_post, on='MEMBER NUMBER', how='left').fillna({
    'UNIQUE_OTHER_HOSPITALS': 0,
    'TOTAL_POST_VISITS': 0
})
member_switches['SWITCHED'] = member_switches['TOTAL_POST_VISITS'] > 0

# 6. Aggregate by FIRST_HOSPITAL – using SUMS, not averages
hospital_stats = member_switches.groupby('FIRST_HOSPITAL').agg(
    PATIENTS_STARTED=('MEMBER NUMBER', 'nunique'),
    PATIENTS_SWITCHED=('SWITCHED', 'sum'),
    TOTAL_POST_VISITS_ALL=('TOTAL_POST_VISITS', 'sum'),          # sum of all post‑switch visits
    TOTAL_UNIQUE_OTHER_HOSP_ALL=('UNIQUE_OTHER_HOSPITALS', 'sum')
).reset_index()

hospital_stats['PATIENTS_RETAINED'] = hospital_stats['PATIENTS_STARTED'] - hospital_stats['PATIENTS_SWITCHED']
hospital_stats['RETENTION_RATE'] = (hospital_stats['PATIENTS_RETAINED'] / hospital_stats['PATIENTS_STARTED']) * 100

# 7. Rank worst by absolute number of switchers
worst = hospital_stats.sort_values('PATIENTS_SWITCHED', ascending=False)
best = hospital_stats.sort_values('PATIENTS_RETAINED', ascending=False)

# 8. Display tables – show actual total post‑switch visits
print("🏥 WORST HOSPITALS – ACTUAL POST‑SWITCH VISIT COUNTS\n")
worst_display = worst[['FIRST_HOSPITAL', 'PATIENTS_STARTED', 'PATIENTS_SWITCHED', 
                       'TOTAL_POST_VISITS_ALL', 'TOTAL_UNIQUE_OTHER_HOSP_ALL', 'RETENTION_RATE']].head(10).copy()
worst_display.columns = ['Hospital', 'Patients Started', 'Patients Switched', 
                         'Total Post‑Switch Visits', 'Total Unique Other Hospitals Visited', 'Retention Rate (%)']
worst_display['Retention Rate (%)'] = worst_display['Retention Rate (%)'].round(1)
display(worst_display)

print("\n🏆 BEST HOSPITALS – Most patients retained\n")
best_display = best[['FIRST_HOSPITAL', 'PATIENTS_STARTED', 'PATIENTS_RETAINED', 'RETENTION_RATE']].head(10).copy()
best_display.columns = ['Hospital', 'Patients Started', 'Patients Retained', 'Retention Rate (%)']
best_display['Retention Rate (%)'] = best_display['Retention Rate (%)'].round(1)
display(best_display)

# 9. Bar chart – total post‑switch visits (not average)
fig_total_visits = px.bar(
    worst.head(10),
    x='TOTAL_POST_VISITS_ALL',
    y='FIRST_HOSPITAL',
    orientation='h',
    title='📊 Total Post‑Switch Visits (sum of all visits after leaving)',
    labels={'TOTAL_POST_VISITS_ALL': 'Total Post‑Switch Visits', 'FIRST_HOSPITAL': ''},
    text='TOTAL_POST_VISITS_ALL',
    color='TOTAL_POST_VISITS_ALL',
    color_continuous_scale='Reds'
)
fig_total_visits.update_traces(texttemplate='%{text}', textposition='outside')
fig_total_visits.update_layout(height=500, margin=dict(l=150))
fig_total_visits.show()

# 10. DETAILED PATIENT‑LEVEL DATA (for the worst hospital, for example)
# Show first 10 switchers from the worst hospital with their actual post‑switch visit counts
worst_hospital = worst.iloc[0]['FIRST_HOSPITAL']
worst_patients = member_switches[member_switches['FIRST_HOSPITAL'] == worst_hospital]
worst_switchers = worst_patients[worst_patients['SWITCHED']][['MEMBER NUMBER', 'TOTAL_POST_VISITS', 'UNIQUE_OTHER_HOSPITALS']]
print(f"\n🔍 Example: Patients who switched from '{worst_hospital}' (actual post‑switch visit counts)")
display(worst_switchers.head(10))

# 11. Optional: histogram of post‑switch visit counts per hospital (choose a few)
import plotly.graph_objects as go
fig_hist = px.histogram(
    member_switches[member_switches['SWITCHED']],
    x='TOTAL_POST_VISITS',
    nbins=20,
    title='Distribution of Post‑Switch Visit Counts (all switchers)',
    labels={'TOTAL_POST_VISITS': 'Number of visits after switch'}
)
fig_hist.show()

🏥 WORST HOSPITALS – ACTUAL POST‑SWITCH VISIT COUNTS



,Hospital,Patients Started,Patients Switched,Total Post‑Switch Visits,Total Unique Other Hospitals Visited,Retention Rate (%)
125,ULINZI PRIME HEALTH SERVICES FUND (UPHSF),1656,421,919.0,518.0,74.6
26,EQUITY AFIA RONGAI,662,188,537.0,243.0,71.6
0,AAR HEALTHCARE NAIROBI,199,65,215.0,98.0,67.3
62,NAIROBI WEST HOSP,150,59,192.0,79.0,60.7
60,NAIROBI HOSP REFERRAL,145,50,171.0,78.0,65.5
117,THE KAREN HOSP REFERRAL,117,46,118.0,62.0,60.7
73,PANDYA MEMORIAL HOSP,86,38,105.0,48.0,55.8
55,MONALIFE PHARMACEUTICALS LTD,82,36,96.0,46.0,56.1
114,THE AGA KHAN HOSP KIS,158,35,126.0,45.0,77.8
96,ST FRANCIS COMMUNITY KASARANI,122,33,85.0,39.0,73.0



🏆 BEST HOSPITALS – Most patients retained



,Hospital,Patients Started,Patients Retained,Retention Rate (%)
125,ULINZI PRIME HEALTH SERVICES FUND (UPHSF),1656,1235,74.6
26,EQUITY AFIA RONGAI,662,474,71.6
0,AAR HEALTHCARE NAIROBI,199,134,67.3
114,THE AGA KHAN HOSP KIS,158,123,77.8
60,NAIROBI HOSP REFERRAL,145,95,65.5
100,ST LUKE ORTHOPEADIC ELD,121,94,77.7
62,NAIROBI WEST HOSP,150,91,60.7
96,ST FRANCIS COMMUNITY KASARANI,122,89,73.0
8,BISHOP KIOKO CATHOLIC,112,88,78.6
105,ST MATIA MULUMBA MISSION,100,85,85.0



🔍 Example: Patients who switched from 'ULINZI PRIME HEALTH SERVICES FUND (UPHSF)' (actual post‑switch visit counts)


,MEMBER NUMBER,TOTAL_POST_VISITS,UNIQUE_OTHER_HOSPITALS
2,DEFMIS-100010-01,3.0,1.0
10,DEFMIS-100028-00,1.0,1.0
11,DEFMIS-100028-01,1.0,1.0
20,DEFMIS-100073-01,1.0,1.0
21,DEFMIS-100083-00,2.0,1.0
22,DEFMIS-100083-01,1.0,1.0
34,DEFMIS-100192-01,1.0,1.0
38,DEFMIS-100198-01,1.0,1.0
57,DEFMIS-100269-00,1.0,1.0
62,DEFMIS-100300-00,5.0,2.0
